# Python functions in practice

A small example of turning messy records into a reliable summary. The emphasis is on pure functions, explicit validation, and tests for edge cases.

In [ ]:
from collections import Counter
from statistics import mean

records = [
    {'name': 'A', 'score': 82, 'submitted': True},
    {'name': 'B', 'score': 91, 'submitted': True},
    {'name': 'C', 'score': None, 'submitted': False},
    {'name': 'D', 'score': 74, 'submitted': True},
]

## 1. Keep responsibilities small

Each function below does one thing and returns a value. That makes the pipeline easy to test, reuse, and replace.

In [ ]:
def submitted_scores(rows):
    """Return numeric scores for records that were submitted."""
    return [row['score'] for row in rows if row.get('submitted') and isinstance(row.get('score'), (int, float))]

def summary(rows):
    scores = submitted_scores(rows)
    if not scores:
        return {'count': 0, 'mean': None, 'highest': None}
    return {'count': len(scores), 'mean': mean(scores), 'highest': max(scores)}

summary(records)

## 2. Add explicit validation

Validation turns silent data errors into a clear failure close to their source.

In [ ]:
def validate_record(row):
    required = {'name', 'score', 'submitted'}
    missing = required - row.keys()
    if missing:
        raise ValueError(f'missing fields: {sorted(missing)}')
    if row['submitted'] and not isinstance(row['score'], (int, float)):
        raise ValueError('submitted records need a numeric score')
    return True

for record in records:
    validate_record(record)
print('all records validated')

## 3. Test the edge cases

Small assertions are useful documentation as well as protection against regressions.

In [ ]:
assert submitted_scores(records) == [82, 91, 74]
assert summary([]) == {'count': 0, 'mean': None, 'highest': None}
try:
    validate_record({'name': 'broken', 'submitted': True, 'score': '91'})
except ValueError as error:
    print('expected validation error:', error)
else:
    raise AssertionError('invalid score should fail validation')

## Takeaway

Functions are most valuable when they make a repeated decision explicit: what counts as a valid record, how missing data is handled, and what summary is returned. The same pattern scales from a four-row example to a production data pipeline.